In [0]:
from pyspark.sql import functions as F

In [0]:
%sql
use catalog medallion_catalog

In [0]:
%sql
SHOW VOLUMES

In [0]:
messy_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "false")
    .csv("/Volumes/medallion_catalog/landing/landing_volume/raw_orders_messy.csv")
)

messy_df.show()

messy_df.printSchema()

In [0]:
messy_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("medallion_catalog.bronze.bronze_orders")

In [0]:
%sql
SELECT * FROM medallion_catalog.bronze.bronze_orders

In [0]:

bronze_df = (
    messy_df
    .withColumn("load_date", F.current_timestamp())
    .withColumn("source_path", F.col("_metadata.file_path"))
    .withColumn("source_file", F.col("_metadata.file_name"))
)

bronze_df.show()

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("medallion_catalog.bronze.bronze_orders")

In [0]:
%sql
SELECT * FROM  medallion_catalog.bronze.bronze_orders

In [0]:
%sql
DESCRIBE HISTORY medallion_catalog.bronze.bronze_orders